In [1]:
import torch
import torchvision
import torchvision.transforms as transforms

print(torch.__version__)

2.11.0+cpu


In [2]:
transform = transforms.Compose([

    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),

    transforms.ToTensor()

])

print(transform)

Compose(
    RandomHorizontalFlip(p=0.5)
    RandomVerticalFlip(p=0.5)
    RandomRotation(degrees=[-30.0, 30.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=None, hue=None)
    ToTensor()
)


In [3]:
from torchvision.datasets import CIFAR10

train_dataset = CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

100%|██████████| 170M/170M [45:38<00:00, 62.2kB/s]


In [4]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32
)

In [5]:
from torchvision.models import resnet50

model = resnet50(weights='DEFAULT')

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features

model.fc = torch.nn.Linear(
    num_features,
    10
)

print(model.fc)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 180MB/s]


Linear(in_features=2048, out_features=10, bias=True)


In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)

criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

In [8]:
epochs = 1

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}, Loss={running_loss:.4f}"
    )

Epoch 1, Loss=3105.4054


In [9]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(
            outputs,
            1
        )

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

accuracy = 100 * correct / total

print("Accuracy =", accuracy)

Accuracy = 38.34


In [10]:
import cv2

cap = cv2.VideoCapture(0)

ret, frame = cap.read()

if ret:
    print("Camera Working")

cap.release()